<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/chip_network_sim/chip_network_sim/dev_journals/kgosine_202603_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Simulation Framework


This simulation framework models an m x n  network of readout chips. Each chip is described at the RTL level and communicates with neighboring chips using a lightweight messaging API. In the current implementation, each chip receives data from a single upstream neighbor, forwards data to a single downstream neighbor, and receives timing information from a shared launcher process. In addition to forwarding packets, each chip generates data locally, with traffic buffered using a single local FIFO.

The network of chip processes is launched by an orchestrator program (orchestrator.c), which is responsible for initializing the array configuration and coordinating global simulation timesteps. At each timestep, the orchestrator advances the global clock and ensures that all chip instances update synchronously.

Inter-chip communication is implemented using nanomsg-next-generation (NNG), a lightweight, brokerless messaging API used to transport data packets between processes. Each chip instance maintains four NNG sockets (specified using TCP URLs) that support communication between neighboring chips, between chips and the orchestrator, and between chips and a metrics collector. Importantly, this messaging occurs across a distributed network of CPUs, where each process represents a single simulated chip.

During the build process, Verilator is used to translate the RTL design into an equivalent C++ model. The resulting compiled model preserves the logical behavior of the hardware and can be evaluated by advancing clock edges within the simulation. This approach allows the distributed software environment to execute hardware-accurate digital models while maintaining high simulation performance.

Taken together, the framework operates as a distributed synchronous hardware simulator. A global clock is maintained by the orchestrator, individual chip processes act as hardware modules, and NNG socket connections serve as the communication links between modules.

## Chip Unit

A chip functions as a 2-input, 1-output unit. The first input is data from one upstream chip. The second input is data generated locally on the chip. Outputs exit the chip's FIFO and are passed to the downstream chip.

The FIFO on the chip is defined via RTL and buffers data trafficked through the chip. It also acts as an arbiter between locally generated and pass-through data (data from the upstream chip).

The locally generated data can be viewed as the output of any analog or mixed-signal front-end structure within the chip which generates data-packets. In this example, this front-end is generalized to a random number generator (defined in software) which generates data packets. The structure of a data packet is as follows:

* 64-bits long
* 16-bit chip_id
* 24-bit timestamp
* 24-bit data payload (randomly generated string)

The rate at which data is locally generated in a chip is configurable via a gen_ppm argument. On every clock TICK, the chip generates a random number between 0-1,000,000 which is compared with the gen_ppm input. If the randomly generated number is less than the gen_ppm number, a data packet is created during that timestep (otherwise no data packet is generated).  

A chip is defined primarily in the script chip_rtl.cpp which does the following:

* take in the Verilated RTL model for the chip (model.cpp)
* communicate with the upstream chip to receive upstream data packet
* pass upstream data packet as input to the chip model (model.cpp)
* (may) generate a local data packet and pass as an input to the chip model (model.cpp)
* simulate one complete time step of the chip model
* communicate with the downstream chip to pass output data packet
* communicate with Orchestrator to receive timestep TICK and send DONE reply
* send performance metrics to a metric collector





<img src="https://drive.google.com/uc?export=view&id=1ftR5xNZ0VA6UkcgqGU0NENhaPZxtJPB4" width="600">



Communication with other chips is handled via NNG sockets which are initialized in the chip.

## Message Transport

<img src="https://drive.google.com/uc?export=view&id=1TkiKIK_jwe_kWmm3c-qoy67zKjI3IqXN" height="450">

The program uses NNG (nanomsg-next-gen) as a lightweight brokerless API to handle the movement of data packets between chips. There are four NNG channels defined per chip:

* Timing Channel: Chip <--> Orchestrator

Used for synchronization with the central clock driver provided by the orchestrator; sends each chip a TICK message which cues the chip simulation to run one clock cycle and reply with a DONE message when complete.

[ Clock (REQ) --TICK--> Chip (REP) ]

Each chip executes one simulation step

[ Chip (REP) --DONE--> Clock (REQ) ]

There is an enforcement to WAIT until *all* chips reply DONE to send the next tick, ensuring synchronization across the network.

* Communication with Upstream Chip: Upstream_Chip <--> Chip

Upon receiving a TICK, the chip will send a request for a data packet from the upstream chip and will wait for a reply from the upstream chip. If the upstream chip has a data packet, it will send it over the reply socket, otherwise it will reply that is has no data packet to transmit.

[ Chip (REQ) --DATA_REQUEST--> Upstream Chip (REP) ]

[ Upstream Chip (REP) --DATA_REPLY--> Chip (REQ) ]

* Communication with Downstream Chip: Chip <--> Downstream Chip

The chip will wait for a request from the downstream chip and then will respond with DATA_REPLY. If the chip has data to send it will transmit over its reply socket, otherwise it will reply that it has no data to send.  

[ Downstream Chip (REQ) --DATA_REQUEST-->  Chip (REP) ]

[ Chip (REP) --DATA_REPLY--> Downstream Chip (REQ) ]

* PUSH socket: metrics channel which sends one-way reporting to a metrics collector.

No blocking or wait for reponse. Used to gather metrics about the simulation. The metrics collector is a function within the orchestrator.


In summary, on each timestep:

* The orchestrator sends TICK to all chips in a network
* The chip will send a request for data to its upstream neighbor and possibly receive data
* The chip will simulate one TICK (one full clock cycle) and publish data to a shared data_server_state_t.
* The chip will wait for a request for data from its downstream neighbor.
* The chip will wait until the main simulation has published its simulation result (which may or may not have generated a packet for output transmission).
* The chip will reply to its downstream neighbor with either an output data packet or that there is no packet to send.
* The chip will send performance metrics to the metrics collector



#### Multithreading


<img src="https://drive.google.com/uc?export=view&id=1qVzSZodUyHX2LKTCeSDKTf9LEXxB_Q3N" height="450">


A single chip unit can (and will, unless the chip is a sink chip) run two threads. The primary thread (aka main simulation thread) does the following:

* receives TICK from orchestrator
* sends DATA_REQ to upstream chip (intiated by this chip and serialized with the TICK)
* receives the DATA_REP from upstream chip
* executes RTL timestep simulation
* sends DONE to orchestrator after simulation step completion
* publishes generated data packet to data_server_state_t

The second thread (aka downstream communication thread) handles communication with the downstream neighbor. This is triggered by the DATA_REQ from the downstream chip which is not necessarily serialized with the tick step and could possibly arrive when the main simulation thread is busy doing its simualtion. The downstream communication thread is opened for every chip with a downstream neighbor (i.e. only the sink chip will not run this thread).

The two threads communicate via data_server_state_t which is shared state object between them. The main simulation thread publishes data packets to this shared data server state. The downstream communication thread will wait to reply to the downstream chip until the main thread has published (either a data packet or that there is no data packet) to the data_server_state_t.

pthread is used to handle the synchronization between these two threads on the same chip. This type of multithreading is done on the same process and uses a shared memory space. This parallelization is in contrast to the way multiple chips are simulated in parallel which is done on a truly distributed computing model. The nng socket communication allows distributed computing across CPUs without shared memory.

## Synchronization

The Orchestrator enforces a strict lock-step synchronization constraint. The Orchestrator sends a TICK(seq) where seq is the timestep count to all chips. All chips then begin executing one model step. Upon completion of the model step, each chip must reply to the Orchestrator with DONE(seq). The Orchestrator will validate that it has received DONE(seq) for all chip_id in the network before advancing to seq+1 and sending another TICK. Because no chip can advance to the next timestep unless all chips have completed and acknowledged the current timestep, the chips in a network are guaranteed to be in global synchronization at every timestep. This is especially important in networks where only some chips are triggered to run a simulation step and others do not recieve any external signal injections. The chips which are idle will return a DONE message while the chips which are triggered are still simulating their timestep. This lock-step constraint ensures that the fast chips do not advance to timesteps ahead of the slow chips and possibly communicate future data to chips simulating previous timesteps. Similarly, for a network simulated over many CPUs of varying speeds, this constraint maintains synchronization across CPUs and processes.


## Packet Loss

If both a locally generated packet and a pass-through packet attempt to enter the FIFO during the same timestep and the FIFO has 2 or more available positions, then both packet types will enter the FIFO during the timestep, with the local packet entering the queue first.

If there is only 1 available position in the FIFO, the locally generated packet will enter the FIFO (causing its occupancy to reach 100%) and the pass-through packet will be dropped.

## Running the Program

The Orchestrator is the top-level simulation driver which parses run settings and launches one chip process per grid cell. It is responsible for running the lock-step control protocol wherein it passes a TICK to all chips and waits to collect DONE from all chips, enforcing that all chips are stepping together in time.

run_from_config.py is an Orchestrator launcher which can pass arguments to the Orchestrator from a configuration file (.json). These confirguration files can define array size, routing, local packet generation rates (per-chip), and fifo depth.

## Example Cases

### 2x2 Array, Single Packet Tracing

A 2x2 array of chips was defined with the following routing:


<img src="https://drive.google.com/uc?export=view&id=1k4TPan2zRxpDxbQG9-Ac1LmwlDUpCaYn" height="300">


For this run, Chip_3 had gen_ppm=1,000,000 (generating local data on each clock tick) and all other chips has gen_ppm=0 (no local packet generation). The passage of the data packet generated at timestep 0 is plotted below.

<img src="https://drive.google.com/uc?export=view&id=11gUIXXqPOtzLXKPXAkTtgh4XxxuuSGR_" height="600">

You can see that the packet is generated in Chip_3 and on the following tick, exits Chip_3 and enters the FIFO of Chip_2 (its downstream neighbor). Because there is only ever one packet in the queue (no other chips generating data and chips only have one upstream neighbor), the packet will exit the chip on the following timestep and enter the FIFO of the downstream chip. This repeats until the packet has exitted the sink chip (Chip_0).

### 3x5 Array, FIFO Overflow

A 3x5 array was defined with the following routing:

<img src="https://drive.google.com/uc?export=view&id=1Xrevq0PNr8tefwGsSCWiJGoBrNwnwhXu" width="600">

With gen_ppm=100,000 for all chips (so local packets are generated 10% of the time). The FIFO depth was 64 deep. FIFO occupancy over 10,000 clock cycles is plotted below.

<img src="https://drive.google.com/uc?export=view&id=1D2pwXOShbbCewylbIVSLVHEwxJgm3oK5" width="1000">

<img src="https://drive.google.com/uc?export=view&id=1lcTUGkzcX88_aRygJXEjs3tX4cGzQnuI" width="1000">

<img src="https://drive.google.com/uc?export=view&id=1VBhulxCgVUNO4UXnqnJRzD2QNmKk9Dhs" width="1000">

These results match well with the prediction for a chain of 15 chips ordered (from source to sink) $ \{14, 13, 12, 11, 10, 5, 6, 7, 8, 9, 4, 3, 2, 1, 0 \}$ (as shown in routing diagram). The probability that a single chip generates a local packet is
$$ p =0.10 $$
The average packet flowing through the i-th chip in a chain is the sum of local traffic from chip i + all chips upstream. The i-th chip in the chain sees an average incoming load of
$$\lambda_i = (15-i)p $$
Such that the expected load for each chip is

| Index in Chain $i$ | Chip ID | Expected Load $\lambda_i$ |
| ----------- | ----------- | -----|
| 0  | 14 | 0.1|
| 1 | 13 | 0.2 |
| 2 | 12 | 0.3|
| 3 | 11 | 0.4 |
| 4  | 10 | 0.5|
| 5 | 5 | 0.6 |
| 6  | 6 | 0.7|
| 7 | 7 | 0.8 |
| 8 | 8 | 0.9|
| 9 | 9 | 1.0 |
| 10  | 4 | 1.1|
| 11| 3 | 1.2 |
| 12 | 2 | 1.3|
| 13 | 1 | 1.4 |
| 14 | 0 | 1.5|

Where the FIFO is only stable (occupancy $\le$ 100%)  if $\lambda_i \le 1$. Therefore, we expect that chips with Chip_ID = 4, 3, 2, 1, 0 (those in the final plot) will, on average, have a FIFO occupancy of 100%.


We can offer an estimate of how many ticks it would take for the first FIFO to overflow by considering two regimes. The first is before any chip is congested and the average pass-through rate to a downstream chip is << 1. Once congestion has occurred at some node in the chain, the pass through rate to the downstream chip is ~1. Chip 4 is the first chip to have an average input greater than 1. If we define persistent backlog as 2-5 packets queued in the FIFO (still significantly less than the depth), the time for this to occur is given by:

$$ t_4 \approx \frac{2 \ \rm{to} \  5}{0.1} = 20 \  \rm{to} \ 50 \ \rm{ticks} $$

Once Chip_4 has on average a non-empty FIFO, it will send, on average, one packet per tick to its downstream chip (Chip_3) and this near constant rate of packets exitting chips will propagate downstream.  

From Chip_4 to Chip_0 there are four downstream steps so that it take approximately $$ 4 \  \rm{steps} \times 20-50 \ \rm{ticks} = 60-150 \ \rm{ticks} $$ for Chip_0 to become congested (FIFO is on average non-empty, average rate of packets exiting chip = 1).

Now that Chip_0 is congested, the number of ticks to fill the FIFO to 100% occupancy is given by considering the rate of growth of FIFO occupancy:

* ~ 1 packet entering per tick
* local traffic generated at rate of 0.1 per tick
* ~ 1 packet exiting the chip per tick

$$\lambda_{\rm{0\_net}} = 1 + 0.1 -1 = 0.1 $$

Such that the number of ticks to overflow Chip_0 is approximately equal to:

$$t_{\rm{Chip0}\_ \rm{overflow}} \approx t_{\rm{Congestion@Chip0}\ } + \frac{\rm{FIFO \ depth}}{\lambda_{\rm{0\_net}}}=(\sim 60 \ \rm{ticks})+\frac{64}{0.1} \approx 700 \ \rm{ticks}$$

For this simulation, the Chip0 FIFO reaches 100% occupancy for the first time after 679 ticks.



We can verify this estimation method by checking the number of ticks to fully occupy the Chip_0 FIFO for the same array, with the same gen_ppm for all chips, and changing only the FIFO depth to 128. We should expect that it would require ~60+1280=1340 ticks. The occupancy as a function of ticks for Chips 4-0 in this case are shown below:


<img src="https://drive.google.com/uc?export=view&id=1AnkYfXMjM0N_5kgai39AO1VWhaNSmHk0" width="900">

In this run, the Chip_0 FIFO reaches 100% occupancy after 1,336 ticks, almost exactly as predicted. For the original FIFO-depth=64 run, traces of the final 2 chips (Chip0 and Chip1) for the final 10 timesteps in the simulation are shown below. You can see that when the FIFO is at occupancy, it will accept the locally generated packet and the pass-through packet will not enter the FIFO. You can also see that late in the simulation (after more than 700 ticks), the last chips in the array are outputting data packets during each timestep.

<img src="https://drive.google.com/uc?export=view&id=1DmiwEPa8_3y4ye_GqrVybH1juEL12rnY" width="900">

## Test Cases


### Determinism

This program is entirely deterministic (given the same seed for the random number generators). The results of running the same simulation of a 3x5 array of chips 15 times shows that the measured behavioral metrics (including number of transmitted packets, transmission timestamps, packet data, packet loss) are identical across all simulations. There is variation in runtime of approx. 2.7% between runs.

#### Determinism Test Report (3x5 Snake BottomRight->TopLeft)

- Config: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/config/network_3x5_determinism_snake_br_to_tl.json`
- Runs: 15
- Deterministic behavior check (`delivered_tx + total_drops + per_chip_drops`): **PASS**
- Full output check (including `cycles_per_sec`): **FAIL**

##### Per-run Results

| Run | Delivered Packets (tx) | Total Drops | Cycles/sec | Per-chip Drops (chip_id:count) | Behavior Match vs Run1 | Full Match vs Run1 |
| --- | ---: | ---: | ---: | --- | --- | --- |
| 1 | 261827 | 0 | 2329.463 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | yes |
| 2 | 261827 | 0 | 2266.484 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 3 | 261827 | 0 | 2304.796 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 4 | 261827 | 0 | 2283.983 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 5 | 261827 | 0 | 2245.862 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 6 | 261827 | 0 | 2291.448 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 7 | 261827 | 0 | 2266.732 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 8 | 261827 | 0 | 2192.259 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 9 | 261827 | 0 | 2233.954 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 10 | 261827 | 0 | 2211.480 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 11 | 261827 | 0 | 2203.634 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 12 | 261827 | 0 | 2131.800 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 13 | 261827 | 0 | 2142.861 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 14 | 261827 | 0 | 2169.355 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 15 | 261827 | 0 | 2166.838 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |

The fastest/slowest runs were 2329.463/2131.800 cycles/sec. Runtime speed varied by ~2.7%, typical for CPU noise.

### FIFO Arbitration Condition (RTL defined)

The RTL for the chip FIFO defines that if a local packet is generated during the same timestep that a pass-through packet enters from the upstream chip, local packet takes priority in entering the FIFO queue. Both packets may enter the FIFO during a single clock tick. The priority condition is evident when there is only one available position in the FIFO such that the local packet should enter and the pass-through packet is dropped.

The test defines a 2 chip network where the downstream chip is the DUT. The gen_ppm of both chips is defined as 1,000,000 such that a local packet is generated each clock tick. The FIFO depth is set to 2. A measure of the local packets dropped versus pass through packets dropped on Chip_1 is recorded.


#### 1x2 Local-Priority Packet-Loss Test Report

- Config: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/config/network_1x2_priority_loss_test.json`
- Trace run: `traces/priority_loss_1x2/run_1772650434_2818623`
- Run log: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/reports/priority_loss_1x2/latest_run.log`

##### Test Setup

- Topology: chip `0 -> 1`, with chip `1` as downstream sink (`out_id=-1`)
- Grid: `1x2` chips
- `gen_ppm`: `1,000,000` for both chips (generate every tick)
- `fifo_depth`: `2`
- `ticks`: `2000`

##### Run-Level Metrics (orchestrator)

- delivered packets (`tx`): 1999
- received packets (`rx`): 1999
- locally generated packets (`local`): 4000
- total drops (`drops`): 1998
- fifo peak occupancy (`fifo_peak`): 2
- cycles/sec: 5220.575

##### Packet Loss by Chip (from trace events)

| Chip | Local Enq OK | Neighbor Enq OK | Local Drops | Neighbor Drops | Total Drops | DEQ_OUT |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 2000 | 0 | 0 | 0 | 0 | 1999 |
| 1 | 2000 | 1 | 0 | 1998 | 1998 | 1999 |

##### Sink Output Provenance (chip 1, DEQ_OUT events)

- Total sink output packets: 1999
- From chip 1 local traffic (`src_id=1`): 1998
- From chip 0 pass-through traffic (`src_id=0`): 1
- `src_id=0` packet `timestamp` values: `0`
- From other sources: 0

##### Finding

- Interpretation: The only time a pass-through packet is able to enter the FIFO of Chip_1 is at timestep 0 when there are two available possitions in the Chip_1 FIFO. For every following tick, there is only one available position in the FIFO which accepts the locally generated packet over the Chip_0 output packet.

### FIFO Queuing

This test tracks the FIFO queue for a single chip in a run. In this example, the 2x2 network was used where Chip 3 sends traffic into Chip 2 which is generating traffic at a rate sufficient to fill the FIFO and begin dropping packets. The FIFO is 5-wide and the queue is shown below. You can see packets in a full FIFO shift by one position in the queue at each timestep.





#### Run Summary

- Run directory: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/reports/single_packet_2x2_br_to_tr_all4/run_1772842223_3038608`
- Manifest format: `chipsim-trace-manifest-v1`
- Ticks: `11`
- FIFO depth used for table columns: `5`
- Selected chip for FIFO table: `2`
- Chip binary: `./build/chip`
- Table generator script: `scripts/export_fifo_contents.py`
- Table generator command: `python3 scripts/export_fifo_contents.py --run-dir /home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/reports/single_packet_2x2_br_to_tr_all4/run_1772842223_3038608 --chip-id 2 --fifo-depth 5 --format md --include-io-cols --out /home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/reports/single_packet_2x2_br_to_tr_all4/run_1772842223_3038608/chip_2_fifo_table_depth5_with_io.md`

#### Routing And Generation

| chip_id | input_id | out_id | gen_ppm |
| --- | --- | --- | --- |
| 0 | 2 | 1 | 0 |
| 1 | 0 | -1 | 0 |
| 2 | 3 | 0 | 1000000 |
| 3 | -1 | 2 | 1000000 |

#### Per-Tick FIFO Table

| tick | enter_local | enter_neigh | exit_out | slot_0 | slot_1 | slot_2 | slot_3 | slot_4 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 0x0002000000CFCF6B | - | - | 0x0002000000CFCF6B | - | - | - | - |
| 1 | 0x0002000001825438 | 0x000300000044B65A | 0x0002000000CFCF6B | 0x0002000001825438 | 0x000300000044B65A | - | - | - |
| 2 | 0x00020000023248A7 | 0x00030000015FA625 | 0x0002000001825438 | 0x000300000044B65A | 0x00020000023248A7 | 0x00030000015FA625 | - | - |
| 3 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x000300000044B65A | 0x00020000023248A7 | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | - |
| 4 | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000023248A7 | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 |
| 5 | 0x00020000058C00CC | - | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC |
| 6 | 0x0002000006042E9C | - | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C |
| 7 | 0x0002000007F57D00 | - | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 |
| 8 | 0x00020000082A26B5 | - | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 |
| 9 | 0x0002000009EA2C3A | - | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 | 0x0002000009EA2C3A |
| 10 | 0x000200000A9932CD | - | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 | 0x0002000009EA2C3A | 0x000200000A9932CD |

## Applications

**Simulation Scale**

A primary motivation for this simulation framework is the large scale of pixelated detector systems such as those proposed for DUNE. For 4 × 4 mm pixels, the channel density is approximately 62,500 pixels per m². Assuming readout chips with 64 channels each, this corresponds to roughly $10^3$ chips per m². Pixelating a single face of a DUNE module, with an area of approximately 380 m², therefore requires on the order of $3.8 \times 10^5$ readout chips. Simulating a communication network of this scale cannot be performed efficiently within a single-process simulation environment. The distributed networking architecture presented here allows simulations to scale to detector-sized systems by distributing chip models across multiple processes while preserving realistic communication behavior.


**Simulation from Hardware Design**

This framework also enables validation of system-level behavior using detailed chip models derived directly from RTL or transistor-level designs. In traditional development workflows, chip designs are often validated at the RTL or SPICE level and then represented in large-scale simulations using simplified behavioral models. Such simplifications can obscure edge cases or emergent behaviors that arise from the interaction of many chips operating simultaneously. In contrast, this architecture allows compiled RTL models to be integrated directly into tile-level simulations without simplifying internal chip logic. Consequently, unintended effects arising from the transistor-level implementation or communication protocol can manifest naturally in the system-level simulation. The architecture is designed to be adaptable to different chip designs and communication protocols. As demonstrated with the current prototype implementation, the simulation framework can incorporate arbitrary chip models compiled using Verilator. This enables rapid integration of new digital designs while maintaining compatibility with the existing simulation infrastructure. Additionally, the communication layer can be modified to support alternative network features, such as acknowledgement (ACK/NACK) signaling or modified routing strategies, allowing different networking schemes to be evaluated with minimal software changes.

**Easy Iteration of Simple Design Parameters**

Allows for study of design parameters by examining network-wide behavior. For example, simulations can quantify packet loss as a function of FIFO depth, data generation rate, routing strategy, or network topology. These studies provide valuable guidance for system optimization and help identify performance limits and failure modes before hardware implementation.

**Integration with Detector Simulation**

Integration with detector simulation tools such as GEANT4 can provide a complete physics-to-bits simulation chain. In this configuration, GEANT4 generates physics events and simulates the resulting energy deposition within the detector medium. The charge deposited on the detector readout plane can then be mapped onto the pixel geometry and used as input stimulus to the simulated readout electronics.

The deposited charge can drive an analog front-end model implemented using SPICE. The resulting signals are then processed by the RTL-defined digital backend implemented in the chip model, which handles buffering and packet formation.

By combining detector simulation with detailed electronics models, the framework enables end-to-end simulations that propagate information from the initial physics interaction through charge generation, analog signal formation, digital packet creation, and ultimately network-level data transport. This capability allows the response of an entire detector system to be studied for individual events or realistic event streams.


## Future Directions


**Incorporation of analog SPICE simulation**

The simulation framework can be extended to include analog behavior through SPICE/RTL co-simulation. Previous work has successfully demonstrated co-simulation between SPICE and RTL models using Verilator. In this configuration, ngspice invokes the Verilator-generated C++ model corresponding to the RTL implementation while an external orchestrator advances the global simulation time.

Within each chip instance, ngspice performs analog updates locally in CPU memory. Evaluation of the digital RTL model occurs only when the analog timestamp coincides with a digital clock edge. At these points, ngspice calls the Verilator-generated C++ model to update the digital state. In this configuration, the chip_rtl.cpp interface functions are reduced to three primary tasks: injecting analog stimulus into a chip instance, invoking ngspice to advance the co-simulation timestep, and managing socket-based communication with neighboring chip simulations.

This approach enables analog effects such as front-end response and threshold behavior to be incorporated into the system-level simulation while maintaining compatibility with the distributed networking architecture.

**Routing with inputs from multiple neighboring chips**

The current implementation assumes a simple daisy-chain routing configuration. While this model is useful for initial studies, it introduces a single point of failure within the communication path. A planned extension is to allow each chip to receive data from any of its four adjacent neighbors. This will enable more robust routing schemes and allow alternative network topologies to be explored. At present, routing decisions would still be defined at the top level of the simulation, but the underlying architecture will support more flexible inter-chip communication patterns.

**Integration of existing RTL designs**

Another planned extension is the integration of existing RTL implementations such as the LArPix architecture. Incorporating a realistic RTL design will allow the simulation to model more accurate packet formation and transmission timing, including buffering behavior, packet loss, and data routing.